In [3]:
import torch
import torchvision.transforms as transforms
import torch.nn as nn
from PIL import Image

ModuleNotFoundError: No module named 'torch'

In [2]:
from torch.nn.modules import padding
class ImageNeuralNetwork(nn.Module):
  def __init__(self,features_num = 6):
    super().__init__()
    self.features= nn.Sequential(nn.Conv2d(3,32,kernel_size=3,padding=1),
                                 nn.ReLU(),
                                 nn.MaxPool2d(2),

                                 nn.Conv2d(32,64,kernel_size=3,padding=1),
                                 nn.ReLU(),
                                 nn.MaxPool2d(2),

                                 nn.Conv2d(64,128,kernel_size=3,padding=1),
                                 nn.ReLU(),
                                 nn.MaxPool2d(2),)
    self.classifier = nn.Sequential(nn.Flatten(),
                                    nn.Linear(128*28*28,256),
                                    nn.ReLU(),
                                    nn.Dropout(0.5),
                                    nn.Linear(256,features_num))

  def forward(self, x):
    out = self.features(x)
    out = self.classifier(out)
    return out

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
model = ImageNeuralNetwork(features_num=6).to(device)

In [6]:
model.load_state_dict(torch.load('/content/hand_sign.pth', map_location=device))

<All keys matched successfully>

In [7]:
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class_names = ['0', '1', '2', '3', '4', '5']

def predict(image_path):
  image = Image.open(image_path).convert('RGB')
  tensor = transform(image).unsqueeze(0).to(device)

  with torch.no_grad():
    output = model(tensor)
    predicted = torch.argmax(output, dim=1).item()
    print(predicted)
    confidence = torch.softmax(output, dim=1)[0][predicted].item()

  print(f"Fingers shown: {class_names[predicted]}  (confidence: {confidence*100:.1f}%)")

In [8]:
predict('/content/test.png')

2
Fingers shown: 2  (confidence: 100.0%)


In [10]:
predict('/content/photo_2026-06-01_06-56-53.jpg')

5
Fingers shown: 5  (confidence: 100.0%)
